# Data Merging — Stage 4 Assembly 03: Assemble Panel

## Input
- `Data/Data_Collection/Final/Stage_5_Model_Ready/01_unioned/panel_stock_daily_engineered.parquet`
- `Data/Data_Collection/Final/Stage_5_Model_Ready/01_unioned/panel_stock_monthly_engineered.parquet`
- `Data/Data_Collection/Final/Stage_5_Model_Ready/01_unioned/agg_market_daily_full_moments.parquet`, `agg_market_monthly_full_moments.parquet` (used only to identify which base factors are "stock-level")
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/agg_means_nan.parquet` (from notebook 02, pre-clip/pre-fill version)
- `Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes/numbered_classified_moment_inventory_long.csv` — taxonomy file, used to identify which base factors are stock-level vs macro-level by their `level` column, replacing the earlier name-stripping approach (see Step 4)
- `lib.config` — `OUT`, `META`, `BINARIES`, `START_DATE`, `END_DATE`, `CLIP`, `DTYPE`, `FFILL_LIMIT`, `ASOF_TOL`

## Purpose
The stock-level counterpart to notebook 02. Builds the single model-ready panel table — one row per `(permno, date)` — by combining the daily and monthly stock panels and layering on macro context from the already-assembled aggregate table. This is the panel-side output that eventually gets fed into the sparse KAN and compared against the aggregate-side spline fits.

## Step 1 — Load and Inspect
Loads both panel tables, sorted stably by `(permno, date)`. Reports row counts, feature counts, and unique `permno` counts for each.

## Step 2 — Gap Repair, Within Stock
Applies `ffill` to fill short gaps, but critically via `groupby('permno').ffill()` rather than a plain `ffill()`. This is called out explicitly as load-bearing: since rows are ordered by `(permno, date)`, an ungrouped forward-fill would silently carry the last value of one stock across the boundary into the first rows of the *next* stock, with no error raised.

Fill limits are expressed in **rows**, not calendar time, because `pandas.ffill(limit=)` is positional and both panel tables hold exactly one row per stock per period — so a limit of e.g. 3 really does mean "3 periods," unlike the weekly union table in notebook 02, which required special handling because its rows don't correspond 1:1 with calendar weeks.

## Step 3 — Expand Panel Monthly onto Panel Daily
Drops `month_end_cap` from the monthly panel (redundant weight column, not needed downstream) and merges monthly features onto the daily panel via `pd.merge_asof(..., by='permno', direction='backward')`.

The `by='permno'` argument is essential and specifically flagged: without it, `merge_asof` would match purely on date and could pair one stock's daily row with a *different* stock's monthly value. Both frames are sorted on the merge key (`date`), not on `permno`, since that's what `merge_asof` requires.

Asserts the row count is unchanged after the merge (a monthly-to-daily backward-fill merge should never create or drop daily rows) and that there's no column-name collision between the daily and monthly panel feature sets.

## Step 4 — Merge Macro Features from the Aggregate
This is the design-critical section of the notebook. The panel deliberately takes **only the macro features** from the aggregate — the ~145 market-level series (VIX, yields, credit spreads, FX, CFTC positioning, jobless claims) — and explicitly **excludes** the ~134 cap-weighted cross-sectional means of the stock factors (`Tax`, `RoE`, etc.).

Two reasons given for the exclusion:
1. **Naming collision.** In the means-only aggregate table, cap-weighted means are named with the bare base factor (`Tax`, not `Tax_cwmean`) — identical to how the panel names its own stock-level columns. Every shared base factor would collide on merge.
2. **Design integrity.** Including the cwmeans would hand the panel 134 inputs that *are themselves* the aggregate's own inputs, blurring the aggregate-vs-panel spline comparison the whole dissertation is structured around. The two pipelines are meant to observe the *same* base factors at *different levels of aggregation* (individual stock vs. cross-sectional summary) — keeping the panel's and aggregate's feature sets disjoint at the stock level keeps that comparison clean. Giving each stock its own cross-sectional context is explicitly noted as a "documented extension," i.e. future work, not part of this design.

**Identifying which base factors are "stock-level" (revised):** originally this was inferred by checking which base factors carry a `_cwmean` column in either full-moments aggregate table — everything else in the means table was treated as macro. This name-stripping approach was found to break for any base factor name shared, coincidentally, by an unrelated stock feature and an unrelated macro feature. `skew_chg_5d` is exactly this case: the *stock* version (a stock's own option-implied skew change) has a `_cwmean` sibling, so it was placed in `stock_base` — which then silently excluded the *macro* `skew_chg_5d` column (the CBOE SKEW index's own 5-day change, a genuinely distinct market-wide tail-risk signal) from the panel's macro feature block, with no error and no visible sign in the printed drop counts.

The notebook now reads `level` directly from the taxonomy (`numbered_classified_moment_inventory_long.csv`), which assigns `stock`/`macro` per row by inspecting where each factor is actually constructed rather than by pattern-matching column names. It also computes and prints `name_collisions` — the set of base-factor names appearing under both `level` values — so any future recurrence of this failure mode is surfaced explicitly in the notebook output rather than discovered later as a silently missing feature. For `skew_chg_5d` specifically, the notebook carries an explicit named exception (documented inline) that keeps the macro column even though the same base-factor name also appears in `stock_base`, and asserts its presence in the final `macro` list.

**Reads `agg_means_nan.parquet`, not `agg_means.parquet`** — explicitly not the already-clipped/zero-filled/float32 version. Taking the finalized version would double-clip values and, more importantly, fold the aggregate's own imputation into the panel's `NaN` pattern, corrupting `panel_nan.parquet`'s status as a faithful record of what was genuinely missing at the panel level.

Drops `target_daily_return` from the macro frame before merging (targets are built separately in notebook 04), asserts no column collisions with the panel, then performs an **inner** merge on `date` — deliberately not a left merge. Two reasons: the aggregate side is already trimmed to `START_DATE..END_DATE` and its last date is 2024-12-30 while the raw panel runs to 2024-12-31, so the inner join simultaneously applies the burn-in trim and end-date trim in one step, and it makes it structurally impossible for a panel row to end up carrying fabricated (non-existent) macro values.

## Step 5 — Finalise
Produces the same two-file pattern used in notebook 02:
- **`panel_nan.parquet`** — saved first, NaN intact, serving directly as the missingness indicator via `.isna()`.
- **`panel.parquet`** — clipped to `±CLIP`, zero-filled, cast to `DTYPE` (float32).

Clipping and filling are done **column by column, in place**, rather than the vectorized `panel[feats] = panel[feats].clip(...)` pattern used in notebook 02 — explicitly because at this table's size, materializing a full float64 intermediate copy of all feature columns would cost roughly 3 GB on top of the existing frame in memory.

**`dlyret` and `dlycap` are explicitly kept at `float64`**, not downcast to `DTYPE` like everything else — because notebook 04 derives `minret_5d_pct` (a target) from `dlyret`, and deriving a target from an already-downcast column is called out as "an avoidable ambiguity" worth preventing here rather than debugging later.

Builds a per-feature fill report (`rep`) tagging each feature's `block` as `'macro'` or `'stock'` and flagging any feature above 5% imputed, mirroring notebook 02's reporting structure.

*Note: the five binary regime indicators (`vix_above_20`, `vix_above_30`, `curve_inverted_2y10y`, `curve_inverted_3m10y`, `credit_stress`) were subsequently dropped from `panel.parquet`/`panel_nan.parquet` and `fill_report_panel.csv` in-place, in a later step (Stage 4 Assembly 07), rather than by re-running this notebook — see that notebook's documentation for the rationale.*

## Step 6 — Imputed Mass Diagnostics
Reports the >5%-imputed feature list, split by macro/stock block, with an explanatory note on *why* panel-level missingness has a fundamentally different character than aggregate-level missingness: cross-sectional aggregation only requires *any* stock to report for a market-level feature to be considered complete, whereas at the individual stock level a missing observation simply stays missing. Specifically calls out that 11 PERMNOs have no OptionMetrics data at all, and PERMNOs 60097 and 90319 are absent from TAQ for nine consecutive in-universe years.

Also reports distribution-by-block summary stats, and separately computes **imputed mass per stock** (not per feature) — explicitly motivated by the distinction that a feature reading "3% missing" could mean either "3% missing everywhere" or "one stock missing entirely," and the second case is strictly worse, since that one stock would read as "exactly average" for its entire history. This computation is restricted to stock-level features only, since macro columns are identical across all stocks and would just dilute the signal.

## Post-Finalise Investigation Cells (uncommented, exploratory diagnostics)
Several further cells dig into the heavy-imputation-stock problem surfaced above:

1. **Lost stocks check** — identifies any `permno` present in the raw daily panel input but absent from the final assembled `panel`, and prints their date range and row count (presumably to confirm any dropped stocks are dropped for a legitimate reason, e.g. falling entirely outside the trimmed date range).
2. **Heavy-imputation stock representation** — computes, per stock, row count, first/last date, and imputation percentage across stock-level features; reports the 15 worst stocks and, for several thresholds (20/30/40/50%), how many stocks exceed each and what share of total panel rows they represent.
3. **Shape of the worst stock's missingness** — for the single worst-imputed stock, splits its features into "100% missing," "0% missing," and "in between" to distinguish "this stock is missing a few features entirely" from "this stock is a little bit missing everywhere."
4. **Missingness by year** — aggregate imputation rate across all stock features, grouped by calendar year, to see whether the missingness problem is concentrated in early years (new data sources not yet available) or is persistent.
5. **Shared-cause investigation for heavily-imputed stocks** — for every stock above 20% imputed, computes the set of features that are 100% missing for that stock, then uses a `Counter` across all such stocks to find which *features* are most commonly 100%-missing across the group — testing whether the heavy-imputation stocks share one root cause (e.g. all missing the same options/TAQ data source) or represent several independent problems.

## Output
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/panel_nan.parquet` — NaN intact, serves as the missingness indicator.
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/panel.parquet` — clipped, zero-filled, float32 (except `dlyret`/`dlycap`, kept float64), model-ready. Binary regime indicators subsequently removed (see Stage 4 Assembly 07).
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/fill_report_panel.csv` — per-feature NaN/clip/block/imputation-flag report. Binary regime indicator rows subsequently removed.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append('../..')
from lib.config import (OUT, META, BINARIES, START_DATE, END_DATE, CLIP,
                        DTYPE, FFILL_LIMIT, ASOF_TOL)

IN_DIR  = OUT / '01_unioned'
ASM_DIR = OUT / '02_assembled'
OUT_DIR = ASM_DIR
pd.set_option('display.width', 200)

THEMES_DIR = Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready/05_themes')

PAN_D = 'panel_stock_daily_engineered'
PAN_M = 'panel_stock_monthly_engineered'

print('=' * 100)
print('LOAD PANEL')
print('=' * 100)

pan_d = (pd.read_parquet(IN_DIR / f'{PAN_D}.parquet')
           .sort_values(['permno', 'date'], kind='stable').reset_index(drop=True))
pan_m = (pd.read_parquet(IN_DIR / f'{PAN_M}.parquet')
           .sort_values(['permno', 'date'], kind='stable').reset_index(drop=True))

meta_d = [c for c in META[PAN_D] if c in pan_d.columns]
meta_m = [c for c in META[PAN_M] if c in pan_m.columns]
feat_d = [c for c in pan_d.columns if c not in meta_d]
feat_m = [c for c in pan_m.columns if c not in meta_m]

print(f'  daily    {len(pan_d):>8,} rows   {len(feat_d):>4} features   '
      f'{pan_d["permno"].nunique()} permnos')
print(f'  monthly  {len(pan_m):>8,} rows   {len(feat_m):>4} features   '
      f'{pan_m["permno"].nunique()} permnos')

# ── GAP REPAIR, WITHIN STOCK ─────────────────────────────────────────────────
# groupby(permno).ffill(), not a plain ffill: rows are ordered by (permno,
# date), so an ungrouped fill would carry the last value of one stock into the
# first rows of the next -- silently, with no error.
#
# Limits are in ROWS because pandas ffill(limit=) is positional. Both panel
# tables hold one row per stock per period, so the numbers are literal here
# (unlike the weekly union table in notebook 02).

print('\n' + '=' * 100)
print('GAP REPAIR - within stock, at native cadence')
print('=' * 100)

for df, feats, tag, freq in [(pan_d, feat_d, PAN_D, 'daily'),
                             (pan_m, feat_m, PAN_M, 'monthly')]:
    before = int(df[feats].isna().sum().sum())
    df[feats] = df.groupby('permno', sort=False)[feats].ffill(limit=FFILL_LIMIT[freq])
    after = int(df[feats].isna().sum().sum())
    print(f'  {tag:<34} {freq:<8} limit={FFILL_LIMIT[freq]}   '
          f'NaN {before:>9,} -> {after:>9,}  ({before - after:,} repaired)')

# ── MONTHLY -> DAILY, PER STOCK ──────────────────────────────────────────────
# merge_asof with by='permno' matches backward within each stock separately.
# Without it, one stock's daily row could match another stock's monthly row.
# Both frames must be sorted on the merge key (date), not on permno.

print('\n' + '=' * 100)
print('EXPAND PANEL MONTHLY TO DAILY')
print('=' * 100)

pan_m_x = pan_m.drop(columns=[c for c in ('month_end_cap',) if c in pan_m.columns])
clash = (set(pan_m_x.columns) & set(pan_d.columns)) - {'permno', 'date'}
assert not clash, f'panel monthly/daily name collision -> {sorted(clash)}'

panel = pd.merge_asof(
    pan_d.sort_values('date', kind='stable'),
    pan_m_x.sort_values('date', kind='stable'),
    on='date', by='permno', direction='backward',
    tolerance=pd.Timedelta(ASOF_TOL['panel_monthly']),
)
assert len(panel) == len(pan_d), 'merge_asof changed the row count'
print(f'  + panel monthly   {panel.shape[1]:>5} cols   '
      f'tolerance {ASOF_TOL["panel_monthly"]}   {len(panel):,} rows')

del pan_d, pan_m, pan_m_x

LOAD PANEL
  daily     525,957 rows    134 features   227 permnos
  monthly    25,194 rows    154 features   227 permnos

GAP REPAIR - within stock, at native cadence
  panel_stock_daily_engineered       daily    limit=5   NaN 5,668,530 -> 5,423,794  (244,736 repaired)
  panel_stock_monthly_engineered     monthly  limit=3   NaN   463,560 ->   434,926  (28,634 repaired)

EXPAND PANEL MONTHLY TO DAILY
  + panel monthly     292 cols   tolerance 100D   525,957 rows


In [7]:
# ── WHAT THE PANEL TAKES FROM THE AGGREGATE ──────────────────────────────────
# MACRO ONLY -- 145 market-level features (VIX, yields, credit spreads, FX,
# CFTC positioning, jobless claims). NOT the 134 cap-weighted cross-sectional
# means of the stock factors.
#
# Those cwmeans are excluded deliberately. In the means-only aggregate they are
# named with the BARE base factor -- `Tax`, not `Tax_cwmean` -- exactly as the
# panel names its own stock-level columns, so every shared base factor would
# collide. More importantly, including them would give the panel 134 inputs
# that ARE the aggregate's inputs, blurring the aggregate-vs-panel spline
# comparison this dissertation is built on. The two pipelines observe the same
# base factors at different levels; keeping the feature sets disjoint keeps
# that statement clean. Cross-sectional context for each stock is a documented
# extension, not part of this design.
#
# Identifying macro: a stock factor is exactly one with a _cwmean column in a
# full-moments table. Everything else in the means table is macro. This is the
# same rule notebook 01 used to verify the two aggregate tables agree.

print('=' * 100)
print('MERGE MACRO FEATURES')
print('=' * 100)

# EDIT (post-Stage-5-numbering): stock_base is now read from the taxonomy's
# 'level' column rather than inferred by stripping '_cwmean' off column
# names. The name-stripping approach is wrong whenever a base_factor name is
# shared between an unrelated stock feature and an unrelated macro feature --
# concretely, 'skew_chg_5d' exists as BOTH the stock's own option-implied
# skew change AND the macro CBOE SKEW index change. String-stripping put
# 'skew_chg_5d' into stock_base because the stock cwmean column exists in
# agg_market_daily_full_moments, which then silently excluded the MACRO
# skew_chg_5d from the panel's macro feature block -- the macro tail-risk
# signal never reached any panel model, with no error and no visible sign
# in the printed drop counts.
#
# The taxonomy's 'level' column is unambiguous per base_factor because it
# was assigned by inspecting where each factor is actually constructed
# (Panel A/B = stock, Panel C/D = macro), not by pattern-matching names.
taxonomy = pd.read_csv(
    THEMES_DIR / 'numbered_classified_moment_inventory_long.csv',
    dtype={'theme_id': str, 'subtheme_id': str}
)
stock_base = set(taxonomy.loc[taxonomy['level'] == 'stock', 'base_factor'])
macro_base_all = set(taxonomy.loc[taxonomy['level'] == 'macro', 'base_factor'])

print(f"  stock-level base factors (from taxonomy)  : {len(stock_base)}")

# Report any base_factor name used by BOTH levels. This is not itself an
# error -- name reuse is fine -- but it is exactly the situation that broke
# silently before, so it is surfaced every run rather than left implicit.
name_collisions = stock_base & macro_base_all
if name_collisions:
    print(f"\n  NOTE: {len(name_collisions)} base_factor name(s) used by both "
          f"stock and macro features -- verify each resolves correctly "
          f"rather than assuming exclusion is correct:")
    for name in sorted(name_collisions):
        rows = taxonomy[taxonomy['base_factor'] == name][
            ['base_factor', 'level', 'moment', 'subtheme_name']]
        print(rows.to_string(index=False))
        print()

mkt = pd.read_parquet(ASM_DIR / 'agg_means_nan.parquet')

# EXCEPTION: 'skew_chg_5d' is excluded from stock_base's filtering role here
# even though it appears in stock_base. The bare column skew_chg_5d in
# mkt.columns is the MACRO CBOE-SKEW-index change (see the NOTE printout
# above) -- a different quantity from the stock-level skew_chg_5d that also
# exists (with moment suffixes) in the full-moments tables. stock_base
# cannot tell these apart because it only tracks names, not which specific
# column a name refers to. Without this exception the macro column is
# silently dropped from the panel's macro feature block: no error, no
# missing-feature warning, just one fewer market-level signal reaching
# every panel model.
STOCK_BASE_NAME_EXCEPTIONS = {'skew_chg_5d'}

macro = [c for c in mkt.columns
         if c != 'date'
         and (c not in stock_base or c in STOCK_BASE_NAME_EXCEPTIONS)]

print(f"  dropped cap-weighted means                : "
      f"{len((set(mkt.columns) & stock_base) - STOCK_BASE_NAME_EXCEPTIONS)}")
print(f"  macro features retained                   : {len(macro)}")

assert 'skew_chg_5d' in macro, (
    "skew_chg_5d still excluded from panel macro features -- fix did not work.")
print(f"  ✓ skew_chg_5d confirmed present in panel's macro feature block")

MERGE MACRO FEATURES
  stock-level base factors (from taxonomy)  : 288

  NOTE: 1 base_factor name(s) used by both stock and macro features -- verify each resolves correctly rather than assuming exclusion is correct:
base_factor level    moment                    subtheme_name
skew_chg_5d stock    cwmean        Volatility Skew Steepness
skew_chg_5d stock     cwstd    Volatility Skew Steepness Std
skew_chg_5d stock    cwskew  Volatility Skew Steepness Shape
skew_chg_5d stock    cwkurt  Volatility Skew Steepness Shape
skew_chg_5d stock    spread Volatility Skew Steepness Spread
skew_chg_5d macro raw_level    Tail Risk Insurance Repricing

  dropped cap-weighted means                : 287
  macro features retained                   : 293
  ✓ skew_chg_5d confirmed present in panel's macro feature block


In [8]:
assert 'skew_chg_5d' in macro, (
    "skew_chg_5d still excluded from panel macro features after the fix -- "
    "the taxonomy-based stock_base did not resolve as expected.")
print(f"  ✓ skew_chg_5d confirmed present in panel's macro feature block")

  ✓ skew_chg_5d confirmed present in panel's macro feature block


In [9]:
# TWO FILES.
#   panel_nan.parquet   NaN intact. This IS the missingness indicator --
#                       recover it at load with .isna(), no extra columns.
#   panel.parquet       clipped, zero-filled, float32. Model-ready.

print('=' * 100)
print('FINALISE')
print('=' * 100)

meta = [c for c in ['permno', 'date', 'dlyret', 'dlycap'] if c in panel.columns]
binaries = [c for c in BINARIES if c in panel.columns]
feats = [c for c in panel.columns if c not in meta + binaries]
print(f'  meta {len(meta)}   binaries {len(binaries)}   features {len(feats)}')

panel.to_parquet(OUT_DIR / 'panel_nan.parquet', index=False)
print(f'  saved panel_nan.parquet   (NaN intact, {panel.shape[1]} cols, '
      f'{len(panel):,} rows)')

nan_before = panel[feats].isna().sum()
n_clipped  = (panel[feats].abs() > CLIP).sum()
total = len(panel) * len(feats)
print(f'  residual NaN : {int(nan_before.sum()):>12,} of {total:,} cells '
      f'({nan_before.sum() / total:.3%})')
print(f'  clipped      : {int(n_clipped.sum()):>12,} cells '
      f'({n_clipped.sum() / total:.3%})')

# Column by column, in place. panel[feats] = panel[feats].clip(...) would
# materialise a full float64 copy -- ~3 GB at this size, on top of the frame.
for c in feats:
    panel[c] = panel[c].clip(-CLIP, CLIP).fillna(0.0).astype(DTYPE)
for c in binaries:
    panel[c] = panel[c].fillna(0.0).astype(DTYPE)

# dlyret stays float64: notebook 04 builds minret_5d from it, and a target
# derived from a downcast column is an avoidable ambiguity.
for c in ('dlyret', 'dlycap'):
    if c in panel.columns:
        panel[c] = panel[c].astype('float64')

assert panel[feats].isna().sum().sum() == 0, 'NaN survived the fill'
panel.to_parquet(OUT_DIR / 'panel.parquet', index=False)
print(f'  saved panel.parquet   ({DTYPE}, {panel.shape[1]} cols, {len(panel):,} rows)')

rep = pd.DataFrame({'dataset': 'panel', 'feature': feats,
                    'n_nan': nan_before.values,
                    'pct_nan': (nan_before / len(panel)).values,
                    'n_clipped': n_clipped.values})
rep['block'] = np.where(rep['feature'].isin(macro), 'macro', 'stock')
rep['flag']  = np.where(rep['pct_nan'] > 0.05, 'IMPUTED >5%', '')
rep.to_csv(OUT_DIR / 'fill_report_panel.csv', index=False)

FINALISE
  meta 4   binaries 5   features 574
  saved panel_nan.parquet   (NaN intact, 583 cols, 436,669 rows)
  residual NaN :    3,210,037 of 250,648,006 cells (1.281%)
  clipped      :    1,310,254 cells (0.523%)
  saved panel.parquet   (float32, 583 cols, 436,669 rows)


In [10]:
print('=' * 100)
print('IMPUTED MASS - features above 5% zero-filled')
print('=' * 100)
print('Panel NaN has a different cause from the aggregate. Cross-sectional')
print('aggregation skips missing stocks, so a market feature is complete when')
print('ANY stock reports; at stock level a missing observation stays missing.')
print('11 PERMNOs have no OptionMetrics data at all; PERMNOs 60097 and 90319')
print('are absent from TAQ for nine consecutive in-universe years.\n')

bad = rep[rep['flag'] != ''].sort_values('pct_nan', ascending=False)
if len(bad):
    print(bad[['block', 'feature', 'n_nan', 'pct_nan', 'n_clipped']].head(40)
          .to_string(index=False, formatters={'pct_nan': '{:.2%}'.format}))
    print(f'\n  {len(bad)} features above 5%   '
          f'(macro {int((bad["block"] == "macro").sum())}, '
          f'stock {int((bad["block"] == "stock").sum())})')
else:
    print('  none -- every feature below 5%')

print('\n' + '-' * 100)
print('DISTRIBUTION BY BLOCK')
print('-' * 100)
for blk, g in rep.groupby('block'):
    print(f'  {blk:<7} n={len(g):>4}   median {g["pct_nan"].median():.3%}   '
          f'p90 {g["pct_nan"].quantile(.90):.3%}   max {g["pct_nan"].max():.3%}   '
          f'>5% {int((g["pct_nan"] > .05).sum())}')

print('\n' + '-' * 100)
print('IMPUTED MASS PER STOCK  (worst 15)')
print('-' * 100)
print('A feature at 3% could be 3% missing everywhere, or one stock missing')
print('entirely. The second is worse: that stock reads "exactly average" for')
print('its whole history. Macro columns are identical across stocks, so this')
print('is computed on stock-level features only.\n')
stock_feats = rep.loc[rep['block'] == 'stock', 'feature'].tolist()
per_stock = (pd.read_parquet(OUT_DIR / 'panel_nan.parquet',
                             columns=['permno'] + stock_feats)
               .groupby('permno')
               .apply(lambda g: g[stock_feats].isna().to_numpy().mean()))
print(per_stock.nlargest(15).to_string(float_format='{:.2%}'.format))
print(f'\n  stocks above 20% imputed: {int((per_stock > 0.20).sum())} of {len(per_stock)}')

print('\n' + '-' * 100)
print('MOST CLIPPED')
print('-' * 100)
print(rep.nlargest(15, 'n_clipped')[['block', 'feature', 'n_clipped', 'pct_nan']]
      .to_string(index=False, formatters={'pct_nan': '{:.2%}'.format}))

IMPUTED MASS - features above 5% zero-filled
Panel NaN has a different cause from the aggregate. Cross-sectional
aggregation skips missing stocks, so a market feature is complete when
ANY stock reports; at stock level a missing observation stays missing.
11 PERMNOs have no OptionMetrics data at all; PERMNOs 60097 and 90319
are absent from TAQ for nine consecutive in-universe years.

block               feature  n_nan pct_nan  n_clipped
stock  MomOffSeason11YrPlus  43537   9.97%       1348
stock               IntanBM  42618   9.76%       1646
stock CompositeDebtIssuance  38087   8.72%       3189
stock                grcapx  37691   8.63%        970
stock       RevenueSurprise  37640   8.62%       2214
stock      AbnormalAccruals  36148   8.28%       1976
stock          DebtIssuance  36136   8.28%          0
stock             HerfAsset  35835   8.21%          0
stock          BetaTailRisk  35250   8.07%        104
stock            Investment  34530   7.91%        784
stock               

C:\Users\Henry\AppData\Local\Temp\ipykernel_16860\1081228622.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g[stock_feats].isna().to_numpy().mean()))


In [11]:
lost = set(pd.read_parquet(IN_DIR / f'{PAN_D}.parquet', columns=['permno','date'])
             .groupby('permno')['date'].max().index) - set(panel['permno'].unique())
d = pd.read_parquet(IN_DIR / f'{PAN_D}.parquet', columns=['permno','date'])
print(d[d['permno'].isin(lost)].groupby('permno')['date'].agg(['min','max','size']))

              min        max  size
permno                            
12079  2004-01-02 2004-12-31   252
16424  2004-01-02 2005-09-30   441
21371  2004-01-02 2006-12-29   755
24046  2004-01-02 2004-12-31   252
24643  2004-01-02 2005-12-30   504
25487  2004-01-02 2005-12-30   504
34746  2004-01-02 2005-12-30   504
45241  2004-01-02 2004-12-31   252
47159  2004-01-02 2004-03-31    62
47941  2004-01-02 2004-12-31   252
52038  2004-01-02 2004-12-31   252
65138  2004-01-02 2004-06-30   124
65883  2004-01-02 2006-12-29   755
75333  2006-01-03 2006-03-31    62
76226  2004-01-02 2006-12-29   755
76557  2004-01-02 2005-12-30   504
77284  2004-01-02 2005-08-12   407
77546  2004-01-02 2006-12-29   755
77605  2004-01-02 2005-12-30   504


In [12]:
pn = pd.read_parquet(OUT_DIR / 'panel_nan.parquet',
                     columns=['permno', 'date'] + stock_feats)

# 1. how much of the panel do the heavy-imputation stocks actually represent?
g = pn.groupby('permno')
summary = pd.DataFrame({
    'n_rows':   g.size(),
    'first':    g['date'].min().dt.date,
    'last':     g['date'].max().dt.date,
    'pct_imp':  g[stock_feats].apply(lambda x: x.isna().to_numpy().mean()),
})
summary['pct_of_panel'] = summary['n_rows'] / len(pn)
print(summary.nlargest(15, 'pct_imp').to_string(
      formatters={'pct_imp': '{:.1%}'.format, 'pct_of_panel': '{:.2%}'.format}))

for t in (0.20, 0.30, 0.40, 0.50):
    s = summary[summary['pct_imp'] > t]
    print(f'  >{t:.0%} imputed: {len(s):>3} stocks, '
          f'{s["pct_of_panel"].sum():.2%} of rows')

# 2. is it "all features half-missing" or "half the features fully missing"?
worst = summary['pct_imp'].idxmax()
per_feat = pn[pn['permno'] == worst][stock_feats].isna().mean()
print(f'\npermno {worst}: features 100% missing = {(per_feat > 0.99).sum()}, '
      f'0% missing = {(per_feat < 0.01).sum()}, in between = '
      f'{((per_feat >= 0.01) & (per_feat <= 0.99)).sum()}')

        n_rows       first        last pct_imp pct_of_panel
permno                                                     
11762      251  2024-01-02  2024-12-30   50.2%        0.06%
86339      251  2022-01-03  2022-12-30   45.1%        0.06%
86111     1760  2018-01-02  2024-12-30   42.3%        0.40%
80100     2013  2011-01-03  2019-12-31   31.4%        0.46%
85592     1004  2021-01-04  2024-12-30   30.8%        0.23%
78916      755  2015-01-02  2017-12-29   29.4%        0.17%
92156      252  2009-01-02  2009-12-31   27.9%        0.06%
90441     1922  2007-08-01  2019-03-19   23.9%        0.44%
14542     2515  2015-01-02  2024-12-30   23.3%        0.58%
12345      504  2013-01-02  2014-12-31   21.4%        0.12%
79237      253  2008-01-02  2008-12-31   19.8%        0.06%
16851      503  2018-01-02  2019-12-31   19.3%        0.12%
13356      504  2013-01-02  2014-12-31   18.6%        0.12%
89954     1112  2007-08-01  2012-12-31   18.2%        0.25%
79057     1257  2017-01-03  2023-12-29  

In [13]:
by_year = (pn.assign(yr=pn['date'].dt.year)
             .groupby('yr')[stock_feats]
             .apply(lambda g: g.isna().to_numpy().mean()))
print(by_year.to_string(float_format='{:.2%}'.format))

yr
2007   1.14%
2008   1.74%
2009   2.20%
2010   2.10%
2011   2.15%
2012   1.97%
2013   2.09%
2014   2.50%
2015   2.95%
2016   3.04%
2017   3.23%
2018   2.87%
2019   3.17%
2020   2.52%
2021   2.53%
2022   2.85%
2023   2.82%
2024   3.16%


In [14]:
heavy = summary[summary['pct_imp'] > 0.20].index.tolist()

full_missing = {}
for p in heavy:
    m = pn[pn['permno'] == p][stock_feats].isna().mean()
    full_missing[p] = set(m[m > 0.99].index)
    print(f'permno {p:<6} {len(full_missing[p]):>3} features 100% missing   '
          f'{summary.loc[p, "first"]} .. {summary.loc[p, "last"]}')
    print(f'   {sorted(full_missing[p])[:10]}')

# do they share a cause, or are there several?
from collections import Counter
shared = Counter(f for s in full_missing.values() for f in s)
print('\nfeatures 100% missing for the most stocks:')
for f, n in shared.most_common(20):
    print(f'  {f:<32} {n} of {len(heavy)} stocks')

permno 11762  135 features 100% missing   2024-01-02 .. 2024-12-30
   ['AM', 'AbnormalAccruals', 'Accruals', 'AnnouncementReturn', 'AssetGrowth', 'BMdec', 'BetaTailRisk', 'BookLeverage', 'CF', 'Cash']
permno 12345   53 features 100% missing   2013-01-02 .. 2014-12-31
   ['BetaTailRisk', 'CompEquIss', 'CompositeDebtIssuance', 'DebtIssuance', 'DelBreadth', 'Herf', 'HerfAsset', 'IntanBM', 'IntanCFP', 'IntanEP']
permno 14542   58 features 100% missing   2015-01-02 .. 2024-12-30
   ['AM', 'AbnormalAccruals', 'Accruals', 'AnnouncementReturn', 'AssetGrowth', 'BMdec', 'BookLeverage', 'CF', 'Cash', 'CashProd']
permno 78916   80 features 100% missing   2015-01-02 .. 2017-12-29
   ['BetaTailRisk', 'DebtIssuance', 'DelBreadth', 'EP', 'Herf', 'HerfAsset', 'TrendFactor', 'bestbiddepth_dollar_tw_to_cap', 'bestofrdepth_dollar_tw_to_cap', 'bid_depth_rel_5d']
permno 80100   88 features 100% missing   2011-01-03 .. 2019-12-31
   ['AM', 'AbnormalAccruals', 'BetaTailRisk', 'CF', 'Cash', 'CashProd', 'CompEq